# Classificação com scikit-learn
## Monitoria 2: do projeto de ponta a ponta à avaliação de classificadores

**Pré-requisito:** Monitoria 1 (Python básico, pandas e matplotlib) · **Ambiente:** Google Colab · **Duração:** 2h30–3h

**Base:** capítulos 2 (*End-to-end Machine Learning project*) e 3 (*Classification*) do livro *Hands-On Machine Learning*, de Aurélien Géron ([notebooks no GitHub](https://github.com/ageron/handson-ml3)).

### Objetivo
Conhecer o scikit-learn, treinar e avaliar classificadores e escolher métricas de acordo com o custo dos erros.

**Pergunta-guia:** como saber se um classificador é bom o bastante para apoiar uma decisão? Acurácia alta basta?

### Roteiro

| Bloco | Conteúdo | Tempo |
|---|---|---|
| 1 | Introdução ao scikit-learn | 10 min |
| 2 | Projeto de ponta a ponta com os dados do Banco Mundial (cap. 2) | 35 min |
| 3 | MNIST e classificador binário | 15 min |
| 4 | Medidas de desempenho e custo dos erros | 45 min |
| — | Intervalo | 10 min |
| 5 | Classificação multiclasse e análise de erros | 20 min |
| 6 | Multilabel e multioutput | 10 min |
| 7 | Em busca de 97%: ajuste de hiperparâmetros e teste final | 15 min |
| 8 | IA e fechamento | 10 min |

As seções marcadas como *opcional* podem ser feitas em casa.

## 0. Preparação

Todas as bibliotecas já estão instaladas no Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

np.random.seed(42)
print("scikit-learn:", sklearn.__version__)

## 1. Introdução ao scikit-learn

O **scikit-learn** (`sklearn`) é a principal biblioteca de machine learning "clássico" em Python. Ela é fácil de aprender porque todos os objetos seguem o mesmo padrão:

- **Estimador:** qualquer objeto que **aprende** com os dados, pelo método `fit(X, y)`. Ex.: `SimpleImputer` aprende a mediana; `LinearRegression` aprende os coeficientes.
- **Transformador:** estimador que também **transforma** dados, com `transform(X)` (ou `fit_transform(X)`, que faz as duas coisas). Ex.: `StandardScaler`, `OneHotEncoder`.
- **Preditor:** estimador que faz **previsões**, com `predict(X)`, e se avalia com `score(X, y)`. Ex.: `LinearRegression`, `RandomForestClassifier`.

Convenções:
- `X` é uma tabela (linhas = observações, colunas = variáveis); `y` é o alvo.
- **Hiperparâmetros** são escolhidos por você na criação do objeto: `SimpleImputer(strategy="median")`.
- **Parâmetros aprendidos** terminam com `_` e só existem depois do `fit`: `imputer.statistics_`, `modelo.coef_`.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

X_mini = np.array([[1.0], [2.0], [np.nan], [4.0]])
y_mini = np.array([2.0, 4.0, 6.0, 8.0])

imputer = SimpleImputer(strategy="median")  # hiperparâmetro
X_mini_completo = imputer.fit_transform(X_mini)
print("Mediana aprendida:", imputer.statistics_)
print("X sem ausentes:", X_mini_completo.ravel())

modelo = LinearRegression()
modelo.fit(X_mini_completo, y_mini)
print("Coeficiente:", modelo.coef_, "| Intercepto:", modelo.intercept_)
print("Previsão para x = 5:", modelo.predict([[5.0]]))

Esse mesmo padrão `fit` / `transform` / `predict` vale para os mais de cem modelos do scikit-learn. Trocar um modelo por outro costuma exigir só uma linha.

## 2. Projeto de ponta a ponta (capítulo 2)

Um projeto de machine learning segue, em geral, estes passos:

1. **Enquadrar o problema:** qual decisão o modelo vai apoiar? Como medir o sucesso?
2. **Obter os dados.**
3. **Separar um conjunto de teste** antes de olhar demais os dados.
4. **Explorar e visualizar** para ganhar intuição.
5. **Preparar os dados:** tratar ausentes, transformar variáveis, codificar categorias, padronizar escalas.
6. **Escolher e treinar modelos**, comparando-os com validação cruzada.
7. **Ajustar** o melhor modelo (hiperparâmetros).
8. **Avaliar uma única vez no teste** e apresentar a solução.
9. **Colocar em produção, monitorar e manter.**

O livro usa preços de imóveis na Califórnia. Vamos percorrer os mesmos passos com os dados da Monitoria 1.

### Regressão × classificação
- **Regressão:** prever um número (expectativa de vida, preço de um imóvel, receita).
- **Classificação:** prever uma categoria (fraude ou não, inadimplente ou não, qual dígito está na imagem).

### 2.1 Enquadrar o problema e obter os dados

**Problema:** prever a expectativa de vida de um país a partir do PIB per capita (em log), do desemprego, da população e da região. É uma **regressão**. Métrica: **RMSE** (raiz do erro quadrático médio), na mesma unidade do alvo — anos de vida.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/liviameinhardt/curso-decisoes-negocios-dados-epge/main/dados/indicadores_banco_mundial.csv")

dados = df[df["ano"] == 2022].dropna(subset=["expectativa_vida"]).copy()
dados["log_pib"] = np.log10(dados["pib_per_capita_usd"])
dados["log_populacao"] = np.log10(dados["populacao"])

print("Países com expectativa de vida em 2022:", len(dados))
display(dados["regiao"].value_counts())

### 2.2 Separar o conjunto de teste

O teste simula dados que o modelo nunca viu e fica guardado até o final. `random_state` fixa o sorteio para que todos obtenham o mesmo resultado.

Com poucos dados, um sorteio puramente aleatório pode deixar alguma região quase fora do teste. O livro resolve isso com **amostragem estratificada**: o teste mantém a mesma proporção de cada grupo. No scikit-learn, basta o argumento `stratify`.

In [ ]:
from sklearn.model_selection import train_test_split

variaveis = ["log_pib", "desemprego_pct", "log_populacao", "regiao"]
X_paises = dados[variaveis]
y_paises = dados["expectativa_vida"]

X_paises_treino, X_paises_teste, y_paises_treino, y_paises_teste = train_test_split(
    X_paises, y_paises, test_size=0.2, stratify=dados["regiao"], random_state=42
)
print("Treino:", X_paises_treino.shape, "| Teste:", X_paises_teste.shape)

proporcoes = pd.DataFrame({
    "Total": dados["regiao"].value_counts(normalize=True),
    "Teste": X_paises_teste["regiao"].value_counts(normalize=True),
}).round(2)
display(proporcoes)

### 2.3 Explorar (só o treino)

In [ ]:
explorar = X_paises_treino.assign(expectativa_vida=y_paises_treino)

display(explorar.drop(columns="regiao").corr()["expectativa_vida"].sort_values(ascending=False).round(2))

fig, ax = plt.subplots(figsize=(8, 5))
for regiao, grupo in explorar.groupby("regiao"):
    ax.scatter(grupo["log_pib"], grupo["expectativa_vida"], label=regiao, alpha=0.7)
ax.set_xlabel("log10 do PIB per capita (US$)")
ax.set_ylabel("Expectativa de vida (anos)")
ax.legend(fontsize=7)
ax.grid(alpha=0.25)
plt.show()

### 2.4 Preparar os dados com um `Pipeline`

Há valores ausentes e uma coluna de texto (`regiao`). Modelos precisam de números sem ausentes. Um **Pipeline** encadeia transformadores e, no fim, um preditor:

- colunas numéricas: preencher ausentes com a mediana (`SimpleImputer`) e padronizar (`StandardScaler`);
- coluna categórica: uma coluna 0/1 para cada região (`OneHotEncoder`).

O `ColumnTransformer` aplica cada transformação às colunas certas. O pipeline **aprende** medianas e escalas só com o treino e aplica as mesmas regras a dados novos — isso evita *data leakage* (vazamento de dados).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numericas = ["log_pib", "desemprego_pct", "log_populacao"]
categoricas = ["regiao"]

preparacao = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
    (OneHotEncoder(handle_unknown="ignore"), categoricas),
)

X_preparado = preparacao.fit_transform(X_paises_treino)
print("Formato depois da preparação:", X_preparado.shape)
print("Colunas:", preparacao.get_feature_names_out())

### 2.5 Treinar e comparar modelos com validação cruzada

Avaliar no próprio treino é otimista: o modelo já viu aqueles dados. A **validação cruzada** (*k-fold cross-validation*) divide o treino em k partes, treina em k−1 e avalia na que ficou de fora, k vezes.

O scikit-learn usa a convenção "maior é melhor", por isso o RMSE vem com sinal negativo.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

modelos = {
    "Regressão linear": make_pipeline(preparacao, LinearRegression()),
    "Árvore de decisão": make_pipeline(preparacao, DecisionTreeRegressor(random_state=42)),
    "Random forest": make_pipeline(preparacao, RandomForestRegressor(random_state=42)),
}

for nome, modelo in modelos.items():
    modelo.fit(X_paises_treino, y_paises_treino)
    rmse_treino = np.sqrt(((modelo.predict(X_paises_treino) - y_paises_treino) ** 2).mean())
    rmse_cv = -cross_val_score(modelo, X_paises_treino, y_paises_treino,
                               scoring="neg_root_mean_squared_error", cv=5)
    print(f"{nome:20s} RMSE no treino = {rmse_treino:.2f} | validação cruzada = {rmse_cv.mean():.2f} (± {rmse_cv.std():.2f})")

**Interprete**
- A árvore de decisão tem erro quase zero no treino, mas muito maior na validação cruzada. Isso é **overfitting**: o modelo decorou o treino.
- Qual modelo generaliza melhor? A diferença é grande em relação ao desvio entre as partes?
- Um erro de ~3 anos é aceitável? Depende da decisão que o modelo apoia.

### 2.6 Ajustar hiperparâmetros com `GridSearchCV`

O **grid search** testa combinações de hiperparâmetros com validação cruzada e guarda a melhor. Para acessar um hiperparâmetro dentro do pipeline, usamos `nomedopasso__hiperparametro`.

In [ ]:
from sklearn.model_selection import GridSearchCV

modelo_rf = make_pipeline(preparacao, RandomForestRegressor(random_state=42))
grade = {
    "randomforestregressor__max_features": [1, 3, 5],
    "randomforestregressor__min_samples_leaf": [1, 3, 5],
}

busca = GridSearchCV(modelo_rf, grade, cv=5, scoring="neg_root_mean_squared_error")
busca.fit(X_paises_treino, y_paises_treino)

print("Melhores hiperparâmetros:", busca.best_params_)
print("RMSE da validação cruzada:", round(-busca.best_score_, 2))

resultados_busca = pd.DataFrame(busca.cv_results_)
display(resultados_busca[["params", "mean_test_score"]]
        .assign(rmse=lambda t: -t["mean_test_score"]).drop(columns="mean_test_score")
        .sort_values("rmse").head())

### 2.7 Importância das variáveis

A random forest informa quanto cada variável contribuiu para as previsões. Isso ajuda a explicar o modelo e a descartar variáveis inúteis — mas importância **não é** efeito causal.

In [ ]:
melhor_modelo = busca.best_estimator_
importancias = pd.Series(
    melhor_modelo[-1].feature_importances_,
    index=melhor_modelo[0].get_feature_names_out(),
).sort_values(ascending=False)
display(importancias.round(3))

### 2.8 Avaliar uma única vez no teste

In [ ]:
from sklearn.metrics import root_mean_squared_error

previsoes_teste = melhor_modelo.predict(X_paises_teste)
print("RMSE no teste:", round(root_mean_squared_error(y_paises_teste, previsoes_teste), 2), "anos")

Se o RMSE no teste for parecido com o da validação cruzada, a estimativa de erro é confiável. **Não** volte a ajustar o modelo para melhorar o teste: isso tornaria o teste otimista.

**Sua vez:** retire `log_populacao` das variáveis (em `variaveis` e `numericas`) e execute a seção de novo. O RMSE da validação cruzada piora?

In [ ]:
# Escreva sua solução aqui

## 3. MNIST e classificador binário (capítulo 3)

O **MNIST** tem 70 mil imagens de dígitos escritos à mão, de 28×28 pixels. Cada pixel vai de 0 (branco) a 255 (preto). É o "olá, mundo" da classificação.

Por que estudar dígitos em um curso de decisões de negócios? Os conceitos são os mesmos de problemas como fraude, inadimplência, churn ou triagem de documentos: **prever uma categoria e decidir o que fazer com os erros**.

O download leva cerca de 30 segundos.

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", as_frame=False, parser="liac-arff")
X, y = mnist.data, mnist.target.astype(np.int64)

print("X:", X.shape)  # 70 mil imagens × 784 pixels (28 × 28)
print("y:", y.shape)
print("Primeiros rótulos:", y[:10])

In [ ]:
def mostrar_digito(imagem):
    plt.imshow(imagem.reshape(28, 28), cmap="binary")
    plt.axis("off")

plt.figure(figsize=(9, 4))
for i in range(40):
    plt.subplot(4, 10, i + 1)
    mostrar_digito(X[i])
    plt.title(y[i], fontsize=9)
plt.tight_layout()
plt.show()

Cada imagem é uma linha de `X` com 784 números: o modelo não "vê" a imagem, vê uma lista de intensidades de pixels.

### Separar treino e teste

O MNIST já vem embaralhado, e as últimas 10 mil imagens são o conjunto de teste oficial. O livro treina com as 60 mil restantes; **para a aula rodar rápido no Colab, vamos treinar com 20 mil**.

In [ ]:
X_train, y_train = X[:20_000], y[:20_000]
X_test, y_test = X[60_000:], y[60_000:]

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Frequência de cada dígito no treino:")
print(pd.Series(y_train).value_counts().sort_index().to_dict())

### Classificador binário: "é 5 ou não é 5?"

Vamos começar simplificando: um detector de 5. O rótulo vira `True` (é 5) ou `False` (não é 5).

Usamos o `SGDClassifier`, um classificador linear treinado por *gradiente descendente estocástico* (*Stochastic Gradient Descent*, SGD). Ele atribui um peso a cada pixel e soma: se a soma passa de um limite, prevê "5".

In [ ]:
from sklearn.linear_model import SGDClassifier

y_train_5 = (y_train == 5)
y_test_5 = (y_test == 5)

sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)

print("Proporção de 5 no treino:", y_train_5.mean().round(3))
print("A primeira imagem é um 5?", sgd_clf.predict([X[0]]))

## 4. Medidas de desempenho

Avaliar um classificador é mais sutil do que avaliar uma regressão. Esta é a parte central da aula.

### 4.1 Acurácia com validação cruzada

**Acurácia** = proporção de previsões corretas.

In [ ]:
acuracias = cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy")
print("Acurácia em cada parte:", acuracias.round(3))

Mais de 95% de acerto! Mas compare com um "modelo" que **sempre** responde "não é 5":

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier(strategy="most_frequent")
print("Acurácia do 'sempre não':",
      cross_val_score(dummy_clf, X_train, y_train_5, cv=3, scoring="accuracy").round(3))

Cerca de 90% sem aprender nada, porque só ~9% das imagens são 5. Em problemas **desbalanceados** (fraude, inadimplência, doenças raras), a acurácia engana. Precisamos olhar *quais* erros o modelo comete.

### 4.2 Matriz de confusão

`cross_val_predict` devolve, para cada imagem do treino, a previsão de um modelo que **não** a viu no treinamento. Com essas previsões "limpas", contamos acertos e erros por tipo.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

print(confusion_matrix(y_train_5, y_train_pred))
ConfusionMatrixDisplay.from_predictions(y_train_5, y_train_pred,
                                        display_labels=["Não é 5", "É 5"])
plt.show()

Como ler (linhas = classe real, colunas = previsão):

|  | Previu "não é 5" | Previu "é 5" |
|---|---|---|
| **Real: não é 5** | Verdadeiro negativo (VN) | **Falso positivo (FP)** |
| **Real: é 5** | **Falso negativo (FN)** | Verdadeiro positivo (VP) |

### 4.3 Precisão, recall e F1

- **Precisão** = VP / (VP + FP): quando o modelo diz "é 5", com que frequência acerta?
- **Recall** (sensibilidade) = VP / (VP + FN): de todos os 5 reais, quantos o modelo encontrou?
- **F1-score**: média harmônica das duas. Só é alto se ambas forem altas.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precisão: ", round(precision_score(y_train_5, y_train_pred), 3))
print("Recall:", round(recall_score(y_train_5, y_train_pred), 3))
print("F1:       ", round(f1_score(y_train_5, y_train_pred), 3))

**Sua vez:** calcule a precisão e o recall "à mão", usando os quatro números da matriz de confusão, e confira com o resultado acima.

In [ ]:
# Dica:
# vn, fp, fn, vp = confusion_matrix(y_train_5, y_train_pred).ravel()
# print(vp / (vp + fp))

**Qual métrica importa mais? Depende da decisão.**

| Situação | Erro mais caro | Priorizar |
|---|---|---|
| Detectar fraude em cartão (com revisão humana) | Deixar passar fraude (FN) | Recall |
| Aprovar crédito automaticamente | Aprovar mau pagador | Precisão do "bom pagador" |
| Filtro de vídeos para crianças | Liberar vídeo impróprio | Precisão do "seguro" |
| Triagem para exame médico | Não encaminhar quem está doente (FN) | Recall |

### 4.4 O threshold de decisão

O classificador calcula uma **pontuação** para cada imagem (`decision_function`) e prevê "5" se ela passa de um **threshold**, que por padrão é 0. Mudar o threshold troca precisão por recall.

In [ ]:
pontuacao = sgd_clf.decision_function([X[0]])
print("Pontuação da primeira imagem:", pontuacao)

for threshold in [0, 3000, 8000]:
    print(f"Threshold {threshold}: prevê 5? {pontuacao[0] > threshold}")

Para escolher o threshold, obtemos as pontuações de todas as imagens (com validação cruzada) e calculamos precisão e recall para cada threshold possível.

In [ ]:
from sklearn.metrics import precision_recall_curve

y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3,
                             method="decision_function")
precisoes, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, precisoes[:-1], label="Precisão")
ax.plot(thresholds, recalls[:-1], label="Recall")
ax.axvline(0, color="gray", linestyle=":", label="Threshold padrão (0)")
ax.set_xlabel("Threshold")
ax.set_xlim(-50_000, 50_000)
ax.legend()
ax.grid(alpha=0.25)
ax.set_title("Precisão e recall conforme o threshold")
plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(recalls, precisoes)
ax.set(xlabel="Recall", ylabel="Precisão", title="Curva precisão × recall")
ax.grid(alpha=0.25)
plt.show()

**Exemplo:** o gestor pede "pelo menos 90% de precisão". Qual threshold usar e quanto recall perdemos?

In [ ]:
idx_90 = (precisoes >= 0.90).argmax()  # primeira posição com precisão >= 90%
threshold_90 = thresholds[idx_90]

y_train_pred_90 = (y_scores >= threshold_90)
print("Threshold:", round(threshold_90))
print("Precisão: ", round(precision_score(y_train_5, y_train_pred_90), 3))
print("Recall:", round(recall_score(y_train_5, y_train_pred_90), 3))

Quando alguém pede "99% de precisão", a pergunta certa é: **com que recall?**

### 4.5 Escolher o threshold pelo custo dos erros

Em negócios, podemos ir além: atribuir um custo a cada tipo de erro e escolher o threshold que **minimiza o custo total**.

Suponha que o "5" seja uma transação fraudulenta: um falso positivo custa R$ 10 (revisão manual) e um falso negativo custa R$ 100 (fraude não detectada).

In [ ]:
custo_fp = 10
custo_fn = 100

thresholds_teste = np.linspace(-30_000, 30_000, 301)
custos = []
for t in thresholds_teste:
    previsto = y_scores >= t
    fp = (previsto & ~y_train_5).sum()
    fn = (~previsto & y_train_5).sum()
    custos.append(fp * custo_fp + fn * custo_fn)

custos = np.array(custos)
melhor_threshold = thresholds_teste[custos.argmin()]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds_teste, custos)
ax.axvline(melhor_threshold, color="red", linestyle="--",
           label=f"Menor custo: threshold ≈ {melhor_threshold:.0f}")
ax.axvline(0, color="gray", linestyle=":", label="Threshold padrão (0)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Custo total (R$)")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

print("Custo com threshold padrão:", custos[np.abs(thresholds_teste).argmin()])
print("Custo com melhor threshold:", custos.min())

**Sua vez:** e se o falso positivo for caro (por exemplo, bloquear o cartão de um bom cliente custa R$ 80)? Altere `custo_fp` e veja para onde o threshold se move. Explique a intuição.

**Atenção:** o threshold é escolhido com as previsões da validação cruzada no treino. O teste continua guardado.

### 4.6 Curva ROC e AUC

A **curva ROC** mostra, para cada threshold, a taxa de verdadeiros positivos (recall) contra a taxa de falsos positivos (proporção dos negativos reais classificados como positivos). Um classificador aleatório fica na diagonal; um perfeito encosta no canto superior esquerdo.

A **AUC** (área sob a curva) resume a curva em um número: 0,5 = aleatório; 1 = perfeito.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds_roc = roc_curve(y_train_5, y_scores)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label="SGD")
ax.plot([0, 1], [0, 1], "k:", label="Aleatório")
ax.set(xlabel="Taxa de falsos positivos", ylabel="Taxa de verdadeiros positivos (recall)",
       title="Curva ROC")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

print("AUC:", round(roc_auc_score(y_train_5, y_scores), 3))

**Regra prática:** com classes desbalanceadas ou quando os falsos positivos importam mais, prefira a curva precisão–recall. A ROC tende a parecer otimista nesses casos, porque há muitos negativos e a taxa de falsos positivos fica pequena.

### 4.7 Comparar com uma random forest

A `RandomForestClassifier` não tem `decision_function`, mas tem `predict_proba`: a probabilidade estimada de cada classe. Usamos a probabilidade de "é 5" como pontuação. O treino leva cerca de 1 minuto.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(random_state=42, n_jobs=-1)
y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3,
                                    method="predict_proba")
y_scores_forest = y_probas_forest[:, 1]  # probabilidade de ser 5

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for nome, scores in [("SGD", y_scores), ("Random forest", y_scores_forest)]:
    p, r, _ = precision_recall_curve(y_train_5, scores)
    axes[0].plot(r, p, label=nome)
    fpr, tpr, _ = roc_curve(y_train_5, scores)
    axes[1].plot(fpr, tpr, label=f"{nome} (AUC = {roc_auc_score(y_train_5, scores):.3f})")

axes[0].set(xlabel="Recall", ylabel="Precisão", title="Precisão × recall")
axes[1].plot([0, 1], [0, 1], "k:", label="Aleatório")
axes[1].set(xlabel="Taxa de falsos positivos", ylabel="Taxa de verdadeiros positivos",
            title="Curva ROC")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

y_pred_forest = y_scores_forest >= 0.5
print("Random forest — precisão:", round(precision_score(y_train_5, y_pred_forest), 3),
      "| recall:", round(recall_score(y_train_5, y_pred_forest), 3),
      "| F1:", round(f1_score(y_train_5, y_pred_forest), 3))

**Interprete**
- Qual curva domina a outra (fica acima em todos os pontos)?
- A random forest é mais lenta e menos interpretável que o modelo linear. Em que situação você ainda escolheria o SGD?
- Uma probabilidade de 0,7 da random forest significa que 70% dos casos com essa pontuação são 5? Não necessariamente: a probabilidade pode precisar de **calibração** (`CalibratedClassifierCV`).

---
## ☕ Intervalo (10 min)

---
## 5. Classificação multiclasse e análise de erros

Agora queremos reconhecer os 10 dígitos.

- Alguns modelos lidam com várias classes naturalmente (random forest, regressão logística, KNN).
- Modelos binários podem ser combinados: **one-versus-rest** (*OvR*: 10 detectores, vence a maior pontuação) ou **one-versus-one** (*OvO*: 45 duelos entre pares). O scikit-learn faz isso automaticamente.

Exemplo com SVM (que usa OvO), treinado em apenas 2 mil imagens porque escala mal com muitos dados:

In [ ]:
from sklearn.svm import SVC

svm_clf = SVC(random_state=42)
svm_clf.fit(X_train[:2000], y_train[:2000])

print("Previsão para a primeira imagem:", svm_clf.predict([X[0]]))
print("Pontuações das 10 classes:", svm_clf.decision_function([X[0]]).round(2))
print("Classes:", svm_clf.classes_)

### Padronizar ajuda

O SGD é sensível à escala das variáveis. Comparamos a acurácia com e sem `StandardScaler`. Leva cerca de 1–2 minutos.

In [ ]:
sgd_multi = SGDClassifier(random_state=42)
sgd_multi_padronizado = make_pipeline(StandardScaler(), SGDClassifier(random_state=42))

print("Sem padronizar:", cross_val_score(sgd_multi, X_train, y_train, cv=3).round(3))
print("Padronizado:   ", cross_val_score(sgd_multi_padronizado, X_train, y_train, cv=3).round(3))

### Análise de erros

Uma matriz de confusão 10×10 mostra **quais dígitos são confundidos com quais**. Normalizamos por linha (cada linha soma 100%) para comparar classes de tamanhos diferentes. No segundo gráfico, os acertos recebem peso zero e só os erros aparecem.

In [ ]:
y_train_pred_multi = cross_val_predict(sgd_multi_padronizado, X_train, y_train, cv=3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred_multi, ax=axes[0],
                                        normalize="true", values_format=".0%")
axes[0].set_title("Matriz de confusão (% da classe real)")

pesos_erro = (y_train_pred_multi != y_train)
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred_multi, ax=axes[1],
                                        sample_weight=pesos_erro,
                                        normalize="true", values_format=".0%")
axes[1].set_title("Só os erros (% dos erros da classe real)")
plt.tight_layout()
plt.show()

**Interprete:** quais confusões são mais frequentes? Vamos olhar exemplos de 3 classificados como 5 e de 5 classificados como 3.

In [ ]:
def mostrar_exemplos(real, previsto, n=10):
    idx = np.where((y_train == real) & (y_train_pred_multi == previsto))[0][:n]
    plt.figure(figsize=(n, 1.4))
    for i, j in enumerate(idx):
        plt.subplot(1, n, i + 1)
        mostrar_digito(X_train[j])
    plt.suptitle(f"Real: {real} | Previsto: {previsto}")
    plt.show()

mostrar_exemplos(3, 5)
mostrar_exemplos(5, 3)

Um modelo linear dá um peso fixo a cada pixel. 3 e 5 diferem por poucos pixels (a ligação do traço superior), então pequenos deslocamentos ou rotações confundem o modelo. Ideias do livro para melhorar: centralizar as imagens ou fazer **data augmentation** com versões deslocadas (exercício extra 2).

**Em negócios, o equivalente é:** abrir os casos errados, entender *por que* o modelo errou e decidir se o problema está nos dados, nas variáveis ou no modelo.

## 6. Multilabel e multioutput

### Multilabel
Um classificador pode prever **vários rótulos ao mesmo tempo**: por exemplo, "é grande (≥ 7)?" e "é ímpar?". Em negócios: etiquetar um cliente com vários perfis, ou um documento com vários assuntos. O `KNeighborsClassifier` (KNN, *k-nearest neighbors*) aceita isso diretamente: ele prevê a partir das imagens de treino mais parecidas.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

y_multilabel = np.c_[y_train >= 7, y_train % 2 == 1]

knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train, y_multilabel)
print("Primeira imagem (um 5): [≥ 7?, ímpar?] =", knn_clf.predict([X[0]]))

Para avaliar, uma opção é a média do F1 de cada rótulo (`average="macro"`). Usamos só 5 mil imagens porque o KNN é lento para prever.

In [ ]:
y_pred_multilabel = cross_val_predict(knn_clf, X_train[:5000], y_multilabel[:5000], cv=3)
print("F1 médio:", round(f1_score(y_multilabel[:5000], y_pred_multilabel, average="macro"), 3))

### Multioutput: remover ruído das imagens

Na classificação **multioutput**, cada rótulo pode ter várias classes. O exemplo do livro: a entrada é uma imagem com ruído e a saída é a imagem limpa — **784 rótulos** (um por pixel), cada um com valores de 0 a 255.

In [ ]:
rng = np.random.default_rng(42)
X_train_ruido = X_train + rng.integers(0, 100, X_train.shape)
X_test_ruido = X_test + rng.integers(0, 100, X_test.shape)

knn_limpeza = KNeighborsClassifier()
knn_limpeza.fit(X_train_ruido, X_train)  # entrada com ruído, saída limpa
imagem_limpa = knn_limpeza.predict([X_test_ruido[0]])

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1); mostrar_digito(X_test_ruido[0]); plt.title("Com ruído")
plt.subplot(1, 2, 2); mostrar_digito(imagem_limpa[0]); plt.title("Limpa pelo modelo")
plt.show()

## 7. Em busca de 97%: ajuste de hiperparâmetros e teste final

O exercício 1 do capítulo pede um classificador com mais de 97% de acurácia no teste, com `KNeighborsClassifier` e `GridSearchCV` — o mesmo ajuste que fizemos no capítulo 2.

Hiperparâmetros do KNN:
- `n_neighbors`: quantos vizinhos consultar;
- `weights`: todos os vizinhos valem igual (`"uniform"`) ou os mais próximos valem mais (`"distance"`).

A busca leva cerca de 2 minutos.

In [ ]:
grade_knn = {"n_neighbors": [3, 4, 5], "weights": ["uniform", "distance"]}

busca_knn = GridSearchCV(KNeighborsClassifier(), grade_knn, cv=3, n_jobs=-1)
busca_knn.fit(X_train, y_train)

print("Melhores hiperparâmetros:", busca_knn.best_params_)
print("Acurácia na validação cruzada:", round(busca_knn.best_score_, 4))

### Comparação final no conjunto de teste

Só agora, com os modelos escolhidos, olhamos o teste — **uma única vez**.

In [ ]:
from sklearn.metrics import accuracy_score

sgd_multi_padronizado.fit(X_train, y_train)
forest_multi = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_train, y_train)

resultados = pd.DataFrame({
    "Modelo": ["SGD linear (padronizado)", "Random forest", "KNN ajustado"],
    "Acurácia no teste": [
        accuracy_score(y_test, sgd_multi_padronizado.predict(X_test)),
        accuracy_score(y_test, forest_multi.predict(X_test)),
        accuracy_score(y_test, busca_knn.predict(X_test)),
    ],
})
display(resultados.round(4))

Com 20 mil imagens de treino, o KNN fica perto de 97%. Com as 60 mil do livro, passa de 97%. **Mais dados** costuma ajudar tanto quanto um modelo melhor.

**Para decidir, a acurácia não basta.** Pense também em:
- **custo dos erros** (quais confusões importam para o negócio?);
- **interpretabilidade** (é preciso explicar a decisão a um cliente ou regulador?);
- **custo de previsão** (o KNN guarda todo o treino e compara cada caso novo com todos eles);
- **estabilidade** (o desempenho se mantém quando os dados mudam?).

## 8. Usar IA nesta aula

**Para aprender** — cole a matriz de confusão do detector de 5:
> Esta é a matriz de confusão de um classificador binário: [cole]. Explique o que cada número significa, calcule precisão e recall passo a passo e dê um exemplo de negócio em que eu priorizaria cada uma. Não use jargão sem explicar.

**Para decidir** — descreva o problema:
> Vou construir um modelo para prever inadimplência de clientes. Uma inadimplência não detectada custa em média R$ 2.000 e recusar um bom cliente custa R$ 300 de margem perdida. Que métrica devo otimizar e como escolher o threshold de decisão? Mostre código em scikit-learn usando `cross_val_predict` e `precision_recall_curve`.

**Para depurar** — o código abaixo foi "gerado por IA" e dá erro. Execute, leia a mensagem e peça ajuda à IA com o prompt da Monitoria 1 (erro completo + célula + o que você esperava).

In [ ]:
# Código com erro: execute e descubra por quê
print(sgd_clf.predict(X[0]))

## 9. Fechamento

Ao terminar, você deve conseguir:
- Reconhecer o padrão `fit` / `transform` / `predict` do scikit-learn.
- Descrever as etapas de um projeto de machine learning e por que o teste fica guardado.
- Montar um `Pipeline`, comparar modelos por validação cruzada e ajustar hiperparâmetros com `GridSearchCV`.
- Explicar por que a acurácia engana em classes desbalanceadas.
- Ler uma matriz de confusão e calcular precisão, recall e F1.
- Escolher um threshold de decisão por meta de precisão ou por custo dos erros.
- Comparar classificadores com curvas precisão–recall e ROC.
- Diferenciar classificação binária, multiclasse, multilabel e multioutput.

### Antes de sair
1. Um conceito que ficou claro.
2. Uma dúvida que ainda tem.
3. Um problema do seu trabalho em que precisão e recall teriam pesos diferentes.

### Exercícios extras (do capítulo 3)
1. **Acima de 97%:** repita a seção 7 com as 60 mil imagens de treino (`X[:60_000]`).
2. **Data augmentation:** crie cópias de cada imagem deslocadas 1 pixel para cima, baixo, esquerda e direita (`scipy.ndimage.shift`), treine o melhor modelo com o conjunto ampliado e compare.
3. **Titanic:** resolva o [problema do Titanic no Kaggle](https://www.kaggle.com/c/titanic) com um pipeline completo (capítulo 2) e as métricas desta aula.
4. **Spam:** construa um classificador de spam (veja a solução no notebook do capítulo 3).

### Referências
- Géron, A. *Hands-On Machine Learning with Scikit-Learn, Keras and TensorFlow* (3ª ed.) — [notebooks](https://github.com/ageron/handson-ml3): `02_end_to_end_machine_learning_project.ipynb` e `03_classification.ipynb`.
- [Guia do usuário do scikit-learn](https://scikit-learn.org/stable/user_guide.html), em especial [métricas de classificação](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics).